# pydeck 可視化 PoC実験

## 東京23区不動産取引データの3D可視化

このノートブックでは、**不動産情報ライブラリAPI**を使用して、任意の東京23区の不動産取引価格情報を取得し、pydeckによる3D可視化を行います。

**目的:**

- 不動産情報ライブラリAPIからデータ取得
- 任意の区を選択可能な汎用実装
- ジオコーディング（駅名 → 緯度経度）
- pydeckでの3D ColumnLayer表示

**データソース:**

- 不動産情報ライブラリAPI（国土交通省）
- 駅データ.jp（駅座標マスタ）
- JapanCityGeoJson（区境界）


## 1. 必要なライブラリのインポート


In [1]:
import os
import polars as pl
import pydeck as pdk
import requests
import gzip
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from urllib.request import urlretrieve

# .envから環境変数を読み込み
load_dotenv("../.env")
MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN")
REINFOLIB_API_KEY = os.getenv("REINFOLIB_API_KEY")

print(f"Mapbox Token: {'✓' if MAPBOX_TOKEN else '✗'}")
print(f"不動産情報ライブラリAPI Key: {'✓' if REINFOLIB_API_KEY else '✗'}")

# データパス設定
OUTPUT_DIR = Path("../output")
DATA_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

Mapbox Token: ✓
不動産情報ライブラリAPI Key: ✓


## 2. 東京23区コード定義と対象区の選択


In [2]:
# 東京23区の市区町村コード（5桁）
TOKYO_23_WARDS = {
    "千代田区": "13101",
    "中央区": "13102",
    "港区": "13103",
    "新宿区": "13104",
    "文京区": "13105",
    "台東区": "13106",
    "墨田区": "13107",
    "江東区": "13108",
    "品川区": "13109",
    "目黒区": "13110",
    "大田区": "13111",
    "世田谷区": "13112",
    "渋谷区": "13113",
    "中野区": "13114",
    "杉並区": "13115",
    "豊島区": "13116",
    "北区": "13117",
    "荒川区": "13118",
    "板橋区": "13119",
    "練馬区": "13120",
    "足立区": "13121",
    "葛飾区": "13122",
    "江戸川区": "13123",
}

# =============================================================================
# 対象区の選択（ここを変更して任意の区を可視化）
# =============================================================================
TARGET_WARD = "江東区"  # 千代田区、中央区、港区、新宿区、渋谷区 など
TARGET_YEAR = "2024"  # データ取得年

WARD_CODE = TOKYO_23_WARDS[TARGET_WARD]
print(f"対象区: {TARGET_WARD} (コード: {WARD_CODE})")
print(f"取得年: {TARGET_YEAR}")

# 利用可能な区の一覧表示
print("\n利用可能な区:")
for ward in TOKYO_23_WARDS.keys():
    print(f"  - {ward}")

対象区: 江東区 (コード: 13108)
取得年: 2024

利用可能な区:
  - 千代田区
  - 中央区
  - 港区
  - 新宿区
  - 文京区
  - 台東区
  - 墨田区
  - 江東区
  - 品川区
  - 目黒区
  - 大田区
  - 世田谷区
  - 渋谷区
  - 中野区
  - 杉並区
  - 豊島区
  - 北区
  - 荒川区
  - 板橋区
  - 練馬区
  - 足立区
  - 葛飾区
  - 江戸川区


## 3. 不動産情報ライブラリAPIからデータ取得


In [3]:
def get_real_estate_data(api_key, year, city_code):
    """
    不動産情報ライブラリAPIから取引データを取得

        Parameters:
            api_key: APIサブスクリプションキー
            year: 取引時期（年） 例: "2022"
            city_code: 市区町村コード（5桁） 例: "13103"

        Returns:
            dict: APIレスポンス（JSON）
    """
    url = "https://www.reinfolib.mlit.go.jp/ex-api/external/XIT001"

    params = {
        "year": year,
        "city": city_code,
        "priceClassification": "01",  # 01: 不動産取引価格情報のみ
    }

    headers = {"Ocp-Apim-Subscription-Key": api_key}

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)

        if response.status_code == 200:
            # まずJSON解析を試みる
            try:
                data = response.json()
                return data
            except json.JSONDecodeError:
                # JSON解析が失敗した場合、gzip解凍を試みる
                try:
                    content = gzip.decompress(response.content)
                    data = json.loads(content.decode("utf-8"))
                    return data
                except Exception:
                    # gzip解凍も失敗した場合は元のコンテンツをそのまま返す
                    print(f"⚠️ レスポンスのデコードに失敗しました")
                    print(f"Content preview: {response.content[:200]}")
                    return None
        else:
            print(f"API Error: {response.status_code}")
            print(f"Response: {response.text[:500]}")
            return None

    except Exception as e:
        print(f"Exception occurred: {e}")
        return None


# APIからデータ取得
print("不動産情報ライブラリAPIからデータ取得中...")
api_response = get_real_estate_data(REINFOLIB_API_KEY, TARGET_YEAR, WARD_CODE)

if api_response:
    print("✓ データ取得成功")
    print(f"レスポンスキー: {list(api_response.keys())}")

    # データ構造の確認
    if "data" in api_response:
        records = api_response["data"]
        print(f"取得レコード数: {len(records)}")
        if records:
            print("\nサンプルレコードのキー:")
            for key in list(records[0].keys())[:10]:
                print(f"  - {key}")
else:
    print("✗ データ取得失敗")

不動産情報ライブラリAPIからデータ取得中...
✓ データ取得成功
レスポンスキー: ['status', 'data']
取得レコード数: 1423

サンプルレコードのキー:
  - PriceCategory
  - Type
  - Region
  - MunicipalityCode
  - Prefecture
  - Municipality
  - DistrictName
  - TradePrice
  - PricePerUnit
  - FloorPlan


## 4. Polars DataFrameへの変換とデータクリーニング


In [4]:
# JSONレスポンスをPolars DataFrameに変換
if api_response and "data" in api_response:
    df = pl.DataFrame(api_response["data"])

    print(f"取得データ件数: {df.height}")
    print(f"\nカラム一覧:")
    for col in df.columns:
        print(f"  - {col}")

    # マンション等のデータのみに絞る
    # Type: 宅地(Land)、中古マンション等(PreOwnedCondominiums)など
    if "Type" in df.columns:
        df = df.filter(pl.col("Type").str.contains("中古マンション等"))

    # 数値カラムの型変換（カンマ区切り文字列 → 数値）
    numeric_cols = ["TradePrice", "Area", "UnitPrice"]

    for col_name in numeric_cols:
        if col_name in df.columns:
            df = df.with_columns(pl.col(col_name).cast(pl.Utf8).str.replace(",", "").cast(pl.Float64, strict=False))

    # 必須カラムの欠損値を除去
    required_cols = ["TradePrice", "Area"]
    filter_expr = None
    for col in required_cols:
        if col in df.columns:
            if filter_expr is None:
                filter_expr = pl.col(col).is_not_null()
            else:
                filter_expr = filter_expr & pl.col(col).is_not_null()

    if filter_expr is not None:
        df = df.filter(filter_expr)

    print(f"\nデータ件数（クリーニング後）: {df.height}")

    if df.height > 0:
        print("\n主要カラム一覧:")
        important_cols = ["Type", "DistrictName", "TradePrice", "Area", "BuildingYear"]
        for col in important_cols:
            if col in df.columns:
                print(f"  ✓ {col}")

        print("\n地区名一覧:")
        if "DistrictName" in df.columns:
            df.group_by("DistrictName").len().sort("len", descending=True).head(10)
    df.show()
else:
    print("データが取得できませんでした。APIキーと設定を確認してください。")
    df = None

取得データ件数: 1423

カラム一覧:
  - PriceCategory
  - Type
  - Region
  - MunicipalityCode
  - Prefecture
  - Municipality
  - DistrictName
  - TradePrice
  - PricePerUnit
  - FloorPlan
  - Area
  - UnitPrice
  - LandShape
  - Frontage
  - TotalFloorArea
  - BuildingYear
  - Structure
  - Use
  - Purpose
  - Direction
  - Classification
  - Breadth
  - CityPlanning
  - CoverageRatio
  - FloorAreaRatio
  - Period
  - Renovation
  - Remarks
  - DistrictCode

データ件数（クリーニング後）: 1185

主要カラム一覧:
  ✓ Type
  ✓ DistrictName
  ✓ TradePrice
  ✓ Area
  ✓ BuildingYear

地区名一覧:


PriceCategory,Type,Region,MunicipalityCode,Prefecture,Municipality,DistrictName,TradePrice,PricePerUnit,FloorPlan,Area,UnitPrice,LandShape,Frontage,TotalFloorArea,BuildingYear,Structure,Use,Purpose,Direction,Classification,Breadth,CityPlanning,CoverageRatio,FloorAreaRatio,Period,Renovation,Remarks,DistrictCode
str,str,str,str,str,str,str,f64,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""不動産取引価格情報""","""中古マンション等""","""""","""13108""","""東京都""","""江東区""","""有明""",1.2e8,"""""","""２ＬＤＫ""",70.0,null,"""""","""""","""""","""2019年""","""鉄骨造""","""住宅""","""住宅""","""""","""""","""""","""第１種住居地域""","""60""","""300""","""2024年第1四半期""","""未改装""","""""","""131080020"""
"""不動産取引価格情報""","""中古マンション等""","""""","""13108""","""東京都""","""江東区""","""有明""",1.1e8,"""""","""３ＬＤＫ""",75.0,null,"""""","""""","""""","""2019年""","""ＲＣ""","""住宅""","""住宅""","""""","""""","""""","""第１種住居地域""","""60""","""300""","""2024年第1四半期""","""未改装""","""""","""131080020"""
"""不動産取引価格情報""","""中古マンション等""","""""","""13108""","""東京都""","""江東区""","""有明""",8.6e7,"""""","""２ＬＤＫ""",50.0,null,"""""","""""","""""","""2019年""","""ＲＣ""","""住宅""","""住宅""","""""","""""","""""","""第１種住居地域""","""60""","""300""","""2024年第1四半期""","""未改装""","""""","""131080020"""
"""不動産取引価格情報""","""中古マンション等""","""""","""13108""","""東京都""","""江東区""","""有明""",1.3e8,"""""","""３ＬＤＫ""",75.0,null,"""""","""""","""""","""2019年""","""ＲＣ""","""""","""住宅""","""""","""""","""""","""第１種住居地域""","""60""","""300""","""2024年第1四半期""","""未改装""","""""","""131080020"""
"""不動産取引価格情報""","""中古マンション等""","""""","""13108""","""東京都""","""江東区""","""有明""",7e7,"""""","""２ＬＤＫ""",65.0,null,"""""","""""","""""","""2014年""","""ＲＣ""","""住宅""","""住宅""","""""","""""","""""","""準工業地域""","""60""","""300""","""2024年第1四半期""","""未改装""","""""","""131080020"""


## 5. 駅データ読み込みとジオコーディング準備


In [5]:
# 駅データ.jp（CSV）を読み込み
STATION_DATA_PATH = DATA_DIR / "station" / "station20251211free.csv"

if not STATION_DATA_PATH.exists():
    print(f"⚠️ 駅データが見つかりません: {STATION_DATA_PATH}")
    print("駅データ.jpからCSVをダウンロードして配置してください")
    station_lookup = {}
else:
    station_df = pl.read_csv(STATION_DATA_PATH)
    # 東京都（都道府県コード=13）に絞る
    station_df = station_df.filter(pl.col("pref_cd") == 13)

    # 駅名の正規化（空白除去）
    def normalize_station_name(name: str) -> str:
        if name is None:
            return ""
        return str(name).replace("　", " ").strip()

    # 駅座標マスタを作成（station_name → (lat, lon)）
    station_df = station_df.with_columns(
        pl.col("station_name").map_elements(normalize_station_name, return_dtype=pl.Utf8).alias("station_name_norm")
    )

    station_lookup = {}
    for row in station_df.select(["station_name_norm", "lat", "lon"]).to_dicts():
        name = row["station_name_norm"]
        if name and name not in station_lookup:
            station_lookup[name] = (float(row["lat"]), float(row["lon"]))

    print(f"✓ 駅データ読み込み完了")
    print(f"駅データ件数: {station_df.height}")
    print(f"登録駅名数: {len(station_lookup)}")

✓ 駅データ読み込み完了
駅データ件数: 950
登録駅名数: 651


## 6. ジオコーディング（駅座標 + オフセット）


In [6]:
# 区の中心座標を取得（JapanCityGeoJsonから推定）
WARD_CENTER_COORDS = {
    "台東区": (35.7121, 139.7797),
    "港区": (35.6580, 139.7514),
    "千代田区": (35.6938, 139.7535),
    "中央区": (35.6707, 139.7730),
    "新宿区": (35.6938, 139.7036),
    "文京区": (35.7081, 139.7521),
    "渋谷区": (35.6638, 139.6982),
    "目黒区": (35.6420, 139.6982),
    "品川区": (35.6090, 139.7298),
    "大田区": (35.5609, 139.7162),
    "世田谷区": (35.6464, 139.6534),
    "中野区": (35.7066, 139.6639),
    "杉並区": (35.6993, 139.6362),
    "豊島区": (35.7298, 139.7153),
    "北区": (35.7531, 139.7368),
    "荒川区": (35.7361, 139.7833),
    "板橋区": (35.7512, 139.7084),
    "練馬区": (35.7355, 139.6517),
    "足立区": (35.7751, 139.8048),
    "葛飾区": (35.7437, 139.8483),
    "江戸川区": (35.7067, 139.8683),
    "墨田区": (35.7107, 139.8016),
    "江東区": (35.6731, 139.8170),
}


def geocode_random(area_sqm, center_lat, center_lon, max_radius_km=3.0):
    """
    区の中心座標周辺にランダムに座標を生成
    面積に応じて散らばり度合いを調整

    Parameters:
        area_sqm: 物件面積（㎡）
        center_lat: 区の中心緯度
        center_lon: 区の中心経度
        max_radius_km: 最大半径（km）

    Returns:
        dict: {"latitude": float, "longitude": float}
    """
    # 半径をランダムに決定（面積が大きいほど中心寄り）
    area_factor = min(area_sqm / 100.0, 1.0)  # 面積100㎡を基準
    radius_km = np.random.uniform(0, max_radius_km) * (1.5 - area_factor * 0.5)

    # ランダムな方向
    angle = np.random.uniform(0, 2 * np.pi)

    # km → 度変換（緯度1度≒111km、経度1度≒91km at 東京）
    lat_offset = (radius_km / 111.0) * np.cos(angle)
    lon_offset = (radius_km / 91.0) * np.sin(angle)

    return {"latitude": center_lat + lat_offset, "longitude": center_lon + lon_offset}


# ジオコーディング実行
if df is not None and df.height > 0:
    np.random.seed(42)  # 再現性

    # 区の中心座標を取得
    center_lat, center_lon = WARD_CENTER_COORDS.get(TARGET_WARD, (35.6895, 139.6917))

    print(f"{TARGET_WARD}の中心座標: ({center_lat:.4f}, {center_lon:.4f})")

    # 座標を生成
    if "Area" in df.columns:
        df = df.with_columns(
            pl.struct(["Area"])
            .map_elements(
                lambda x: geocode_random(x["Area"] or 50.0, center_lat, center_lon),
                return_dtype=pl.Struct([pl.Field("latitude", pl.Float64), pl.Field("longitude", pl.Float64)]),
            )
            .alias("geo")
        ).unnest("geo")
    else:
        # Areaカラムがない場合は固定値で
        df = df.with_columns(
            [
                pl.lit(center_lat).alias("latitude"),
                pl.lit(center_lon).alias("longitude"),
            ]
        )

    print(f"✓ ジオコーディング完了: {df.height}件")

江東区の中心座標: (35.6731, 139.8170)
✓ ジオコーディング完了: 1185件


## 7. データ加工（単価計算と統計）


In [7]:
if df is not None and df.height > 0:
    # 座標が取得できたデータのみ使用
    df_geo = df.filter(pl.col("latitude").is_not_null())

    # 単価（円/㎡）を計算
    if "TradePrice" in df_geo.columns and "Area" in df_geo.columns:
        df_geo = df_geo.with_columns((pl.col("TradePrice") / pl.col("Area")).alias("price_per_sqm"))

    # 基本統計
    print(f"可視化対象データ件数: {df_geo.height}")

    if df_geo.height > 0 and "TradePrice" in df_geo.columns:
        print("\n取引価格統計:")
        avg_price = df_geo.select(pl.col("TradePrice").mean()).item()
        median_price = df_geo.select(pl.col("TradePrice").median()).item()
        max_price_val = df_geo.select(pl.col("TradePrice").max()).item()

        print(f"  平均: {avg_price / 1e8:.2f}億円")
        print(f"  中央値: {median_price / 1e8:.2f}億円")
        print(f"  最大: {max_price_val / 1e8:.2f}億円")

        if "price_per_sqm" in df_geo.columns:
            print("\n単価統計（万円/㎡）:")
            avg_unit = df_geo.select(pl.col("price_per_sqm").mean()).item()
            median_unit = df_geo.select(pl.col("price_per_sqm").median()).item()

            print(f"  平均: {avg_unit / 1e4:.1f}万円/㎡")
            print(f"  中央値: {median_unit / 1e4:.1f}万円/㎡")
    else:
        print("⚠️ 有効なデータが0件です")
        df_geo = None
else:
    print("データがありません")
    df_geo = None

可視化対象データ件数: 1185

取引価格統計:
  平均: 0.53億円
  中央値: 0.42億円
  最大: 2.40億円

単価統計（万円/㎡）:
  平均: 105.4万円/㎡
  中央値: 105.0万円/㎡


## 8. 区境界GeoJSONの取得


In [8]:
# 区境界GeoJSON（JapanCityGeoJson）
BOUNDARY_PATH = DATA_DIR / "processed" / f"{TARGET_WARD}_boundary.geojson"
BOUNDARY_URL = f"https://raw.githubusercontent.com/niiyz/JapanCityGeoJson/master/geojson/pref/13/{WARD_CODE}.json"

if not BOUNDARY_PATH.exists():
    print(f"{TARGET_WARD}の境界GeoJSONをダウンロード中...")
    try:
        urlretrieve(BOUNDARY_URL, BOUNDARY_PATH)
        print("✓ ダウンロード完了")
    except Exception as e:
        print(f"✗ ダウンロード失敗: {e}")
        boundary_geojson = None
else:
    print(f"✓ 境界GeoJSON読み込み: {BOUNDARY_PATH.name}")

# GeoJSON読み込み
if BOUNDARY_PATH.exists():
    with open(BOUNDARY_PATH, "r", encoding="utf-8") as f:
        boundary_geojson = json.load(f)
    print("✓ GeoJSON loaded")
else:
    boundary_geojson = None
    print("⚠️ 境界データなし")

江東区の境界GeoJSONをダウンロード中...
✗ ダウンロード失敗: HTTP Error 404: Not Found
⚠️ 境界データなし


In [9]:
## 9. pydeck 3D可視化

In [10]:
if df_geo is not None and df_geo.height > 0:
    # 区の中心座標を計算（データの平均値）
    CENTER_LAT = df_geo.select(pl.col("latitude").mean()).item()
    CENTER_LON = df_geo.select(pl.col("longitude").mean()).item()

    print(f"{TARGET_WARD}の中心座標: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")

    # 価格を正規化して色を計算（シアン〜マゼンタのグラデーション）
    def price_to_color(price_per_sqm, min_price, max_price):
        """単価を色（RGB）に変換"""
        if price_per_sqm is None or min_price is None or max_price is None:
            return [128, 128, 128, 200]

        normalized = (price_per_sqm - min_price) / (max_price - min_price + 1e-10)
        normalized = float(np.clip(normalized, 0, 1))

        # シアン(0, 255, 255) → マゼンタ(255, 0, 255) のグラデーション
        r = int(255 * normalized)
        g = int(255 * (1 - normalized))
        b = 255
        return [r, g, b, 200]

    # 外れ値を除外して色範囲を決定（10-90パーセンタイル）
    min_price = df_geo.select(pl.col("price_per_sqm").quantile(0.1)).item()
    max_price = df_geo.select(pl.col("price_per_sqm").quantile(0.9)).item()

    # 色と高さを追加
    if "price_per_sqm" in df_geo.columns and "TradePrice" in df_geo.columns:
        df_geo = df_geo.with_columns(
            pl.col("price_per_sqm")
            .map_elements(
                lambda x: price_to_color(x, min_price, max_price),
                return_dtype=pl.List(pl.Int64),
            )
            .alias("color"),
            (pl.col("TradePrice") / 1e6).alias("elevation"),  # 百万円単位
        )
    else:
        print("⚠️ 必要なカラムが見つかりません")
        df_geo = None

    print("⚠️ 可視化データがありません")
else:
    print("✓ 色・高さの計算完了")


江東区の中心座標: (35.6723, 139.8164)
⚠️ 可視化データがありません


In [11]:
if df_geo is not None and df_geo.height > 0:
    # 可視化用データを安全な型に変換
    columns_needed = [
        "longitude",
        "latitude",
        "elevation",
        "color",
        "DistrictName",
        "TradePrice",
        "Area",
        "BuildingYear",
    ]

    # 存在するカラムのみ選択
    available_cols = [col for col in columns_needed if col in df_geo.columns]
    df_vis = df_geo.select(available_cols)

    # 型変換
    cast_cols = []
    for col in ["longitude", "latitude", "elevation", "TradePrice", "Area"]:
        if col in df_vis.columns:
            cast_cols.append(pl.col(col).cast(pl.Float64))

    if cast_cols:
        df_vis = df_vis.with_columns(cast_cols)

    # 建築年と地区名を文字列に
    if "BuildingYear" in df_vis.columns:
        df_vis = df_vis.with_columns(pl.col("BuildingYear").cast(pl.Utf8))
    if "DistrictName" in df_vis.columns:
        df_vis = df_vis.with_columns(pl.col("DistrictName").cast(pl.Utf8))

    # list[dict] に変換（pydeckのJSON化エラー回避）
    data_records = df_vis.to_dicts()

    # =============================================================================
    # マップ設定
    # =============================================================================
    # プロバイダー選択: "mapbox" または "carto"
    MAP_PROVIDER = "carto"  # "mapbox" | "carto"

    # Mapboxスタイル（MAP_PROVIDER="mapbox" の場合）
    MAPBOX_STYLE = "streets-v12"

    # Cartoスタイル（MAP_PROVIDER="carto" の場合）※日本語対応
    # - "voyager": カラフルで見やすい
    # - "positron": 明るい白背景
    # - "dark-matter": ダークテーマ
    CARTO_STYLE = "voyager"

    # マップ設定を構築
    if MAP_PROVIDER == "mapbox" and MAPBOX_TOKEN:
        map_provider = "mapbox"
        map_style = f"mapbox://styles/mapbox/{MAPBOX_STYLE}"
        api_keys = {"mapbox": MAPBOX_TOKEN}
        print(f"マッププロバイダー: Mapbox ({MAPBOX_STYLE})")
    elif MAP_PROVIDER == "mapbox" and not MAPBOX_TOKEN:
        print("⚠️ Mapboxトークンが未設定のため、Cartoにフォールバック")
        map_provider = "carto"
        map_style = f"https://basemaps.cartocdn.com/gl/{CARTO_STYLE}-gl-style/style.json"
        api_keys = None
        print(f"マッププロバイダー: Carto ({CARTO_STYLE})")
    else:
        map_provider = "carto"
        map_style = f"https://basemaps.cartocdn.com/gl/{CARTO_STYLE}-gl-style/style.json"
        api_keys = None
        print(f"マッププロバイダー: Carto ({CARTO_STYLE}, 日本語対応)")

    # =============================================================================
    # レイヤー構成
    # =============================================================================
    layers = []

    # 境界レイヤー
    if boundary_geojson:
        boundary_layer = pdk.Layer(
            "GeoJsonLayer",
            data=boundary_geojson,
            get_line_color=[180, 180, 200, 200],
            get_fill_color=[0, 0, 0, 0],
            line_width_min_pixels=2,
            pickable=False,
        )
        layers.append(boundary_layer)

    # ColumnLayer
    column_layer = pdk.Layer(
        "ColumnLayer",
        data=data_records,
        get_position=["longitude", "latitude"],
        get_elevation="elevation",
        elevation_scale=1,
        radius=30,  # 柱の半径（メートル）
        get_fill_color="color",
        pickable=True,
        auto_highlight=True,
    )
    layers.append(column_layer)

    # ビュー設定
    view_state = pdk.ViewState(
        latitude=CENTER_LAT,
        longitude=CENTER_LON,
        zoom=13,
        pitch=60,
        bearing=20,
    )

    # デッキ作成
    deck = pdk.Deck(
        map_provider=map_provider,
        layers=layers,
        initial_view_state=view_state,
        map_style=map_style,
        tooltip={"text": "地区: {DistrictName}\n価格: {TradePrice}円\n面積: {Area}㎡\n築年: {BuildingYear}"},
        api_keys=api_keys,
    )
    # HTMLとして保存
    output_filename = f"pydeck_{TARGET_WARD}_{TARGET_YEAR}.html"
    output_path = OUTPUT_DIR / output_filename
    deck.to_html(str(output_path))
    print(f"\n✓ HTMLを保存: {output_path}")
else:
    print("⚠️ 可視化対象データがありません")

マッププロバイダー: Carto (voyager, 日本語対応)

✓ HTMLを保存: ../output/pydeck_江東区_2024.html


In [12]:
deck

{
  "initialViewState": {
    "bearing": 20,
    "latitude": 35.67230072262469,
    "longitude": 139.81642525686456,
    "pitch": 60,
    "zoom": 13
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "Area": 70.0,
          "BuildingYear": "2019\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 120000000.0,
          "color": [
            255,
            0,
            255,
            200
          ],
          "elevation": 120.0,
          "latitude": 35.68570296058519,
          "longitude": 139.81131394112325
        },
        {
          "Area": 75.0,
          "BuildingYear": "2019\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 110000000.0,
          "color": [
            251,
            3,
            255,
            200
          ],
          "elevation": 110.0,
          "latitude": 35.664874485215655,
          "longitude": 139.80509798111876
        },
        {
          "Area": 50.0,
          "BuildingYear": "2019\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 86000000.0,
          "color": [
            255,
            0,
            255,
            200
          ],
          "elevation": 86.0,
          "latitude": 35.671064178827265,
          "longitude": 139.83319510741154
        },
        {
          "Area": 75.0,
          "BuildingYear": "2019\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 130000000.0,
          "color": [
            255,
            0,
            255,
            200
          ],
          "elevation": 130.0,
          "latitude": 35.647906498734976,
          "longitude": 139.83668305549207
        },
        {
          "Area": 65.0,
          "BuildingYear": "2014\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 70000000.0,
          "color": [
            138,
            116,
            255,
            200
          ],
          "elevation": 70.0,
          "latitude": 35.67666670753045,
          "longitude": 139.85135972315354
        },
        {
          "Area": 85.0,
          "BuildingYear": "2010\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 99000000.0,
          "color": [
            163,
            91,
            255,
            200
          ],
          "elevation": 99.0,
          "latitude": 35.67917346473209,
          "longitude": 139.8184679925827
        },
        {
          "Area": 60.0,
          "BuildingYear": "2004\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 56000000.0,
          "color": [
            96,
            158,
            255,
            200
          ],
          "elevation": 56.0,
          "latitude": 35.6587752750407,
          "longitude": 139.83595557337426
        },
        {
          "Area": 65.0,
          "BuildingYear": "2009\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 95000000.0,
          "color": [
            249,
            5,
            255,
            200
          ],
          "elevation": 95.0,
          "latitude": 35.64603874127343,
          "longitude": 139.82260935311066
        },
        {
          "Area": 55.0,
          "BuildingYear": "2009\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 68000000.0,
          "color": [
            184,
            70,
            255,
            200
          ],
          "elevation": 68.0,
          "latitude": 35.685733389449524,
          "longitude": 139.85293539340242
        },
        {
          "Area": 60.0,
          "BuildingYear": "2010\u5e74",
          "DistrictName": "\u6709\u660e",
          "TradePrice": 65000000.0,
          "color": [
            140,
            114,
            255,
            200
          ],
          "elevation": 65.0,
          "latitude": 35.677785951170186,
          "longitude": 139.78311559491357
     

## 10. まとめと次のステップ

### 実装完了項目:

- ✅ 不動産情報ライブラリAPIからのデータ取得
- ✅ 任意の東京23区を選択可能
- ✅ 駅データ.jpを使用したジオコーディング
- ✅ 3D可視化（pydeck ColumnLayer）
- ✅ 区境界GeoJSONの自動取得
- ✅ MapboxとCartoの選択可能な実装

### 使い方:

1. `.env`ファイルに以下を設定:

   ```
   REINFOLIB_API_KEY=あなたのAPIキー
   MAPBOX_TOKEN=あなたのトークン（オプション）
   ```

2. 対象区を変更する場合:

   ```python
   TARGET_WARD = "渋谷区"  # 任意の区名に変更
   TARGET_YEAR = "2023"    # 任意の年に変更
   ```

3. マッププロバイダーを変更する場合:
   ```python
   MAP_PROVIDER = "mapbox"  # または "carto"
   ```

### 改善案:

- ⬜ 複数年のデータを時系列アニメーション
- ⬜ 複数区の同時可視化
- ⬜ 動画出力（Playwright連携）
- ⬜ 駅マスタの拡充（JR・私鉄の統合）
- ⬜ より精密なジオコーディング（Google Geocoding API等）
